# Select lambda on dev, then evaluate once on test  ·  Person B

`nli_rerank` blends a frozen embedder's cosine with a *directional* NLI
judgement:

```
nli_score(a, b)     = P(entailment | a, b) - P(contradiction | a, b)
combined(a, b, lam) = (1 - lam) * cosine(a, b) + lam * nli_score(a, b)
```

Until now `lam` defaulted to 1.0 — pure NLI, chosen by nobody, and a setting
under which the embedding model contributes nothing at all to the score. This
notebook replaces that guess with a selection procedure, run independently for
each of the four frozen embedders (`multilingual-e5`, `labse`,
`alephbert-sentence`, `sambert`).

**Section 4 (dev) selects lambda. Section 6 (test) evaluates it. Nothing between
them may go backwards.** The dev stage never opens a test file — `lambda_sweep`
raises if you hand it one — and the test stage has no code path that writes a
selection. That ordering is the only thing that makes the final numbers mean
what they say, and it cannot be recovered after the fact by promising it held.

The NLI checkpoint is the one `02_train_nli.ipynb` produced on **clean HebNLI**,
read-only. Nothing here trains or modifies it.

Why the rule is what it is: `LAMBDA_SELECTION.md` in the repo root.

## Where everything ends up

| what | where | survives a reset? |
|---|---|---|
| cloned repo | VM disk | no |
| probe splits + Hebrew STS-B | in the repo, cloned with it | no |
| our NLI checkpoint | **Drive**, read-only | yes |
| the four frozen embedders | HF hub, cached on VM disk | no |
| `results/nli_lambda_dev.csv` | VM disk, then downloaded | via git |
| `results/nli_selected_lambdas.json` | VM disk, then downloaded | via git |
| `results/nli_lambda_test.csv` | VM disk, then downloaded | via git |

Nothing here writes to `checkpoints/` on Drive.

## 1. Setup

**Prints** &nbsp; Nothing. This cell only defines a helper.

**Writes** &nbsp; Nothing.

In [ ]:
# Secrets three ways: Colab's store, then the environment, then a prompt. The VS Code
# extension cannot reach Colab's secret store, so the fallbacks are what make this
# notebook portable. getpass also keeps the token out of the saved output.
import os, subprocess, getpass

def get_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            print(f'{name}: from Colab secrets')
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name)
    if value:
        print(f'{name}: from environment')
        return value.strip()
    return getpass.getpass(f'{name}: ').strip()

**Prints** &nbsp; Python and torch versions, the GPU name, and which `google.colab` modules import. `cuda: True` is worth confirming before four sentence-transformers and an NLI model get downloaded.

**Writes** &nbsp; Nothing.

In [ ]:
# What are we running on? Answers 'will this work here' before anything slow.
import platform
print('python      ', platform.python_version())
print('cwd         ', os.getcwd())
try:
    import torch
    print('torch       ', torch.__version__, '| cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu         ', torch.cuda.get_device_name(0))
except ImportError:
    print('torch        not installed yet')
for mod in ('google.colab.userdata', 'google.colab.drive', 'google.colab.files'):
    try:
        __import__(mod)
        print(f'{mod:24s} available')
    except Exception as exc:
        print(f'{mod:24s} NOT available ({type(exc).__name__})')

**Prints** &nbsp; `GH_TOKEN:` and where it came from, pip's log, then the last 3 commits — the top one should be the newest `nli:` commit.

**Writes** &nbsp; The repo at `/content/hebrew-negation-embeddings` on the VM. The working directory moves into it, so every path after this is relative to the repo root.

In [ ]:
OWNER, REPO, BRANCH = 'ItayBoros', 'hebrew-negation-embeddings', 'main'

gh_token = get_secret('GH_TOKEN')
url = f'https://{gh_token}@github.com/{OWNER}/{REPO}.git'
bare_url = f'https://github.com/{OWNER}/{REPO}.git'

if not os.path.exists(REPO):
    subprocess.run(['git','clone','-q','--branch',BRANCH,url,REPO], check=True)
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else '.')

# Re-authenticate origin before every fetch, not just on first clone: the repo is
# private, and the last step below strips the token, so a second run in the same
# session would otherwise fetch unauthenticated and fail.
subprocess.run(['git','remote','set-url','origin', url], check=True)
subprocess.run(['git','fetch','-q','origin',BRANCH], check=True)

# reset --hard, not pull: this VM's checkout is scratch space that always mirrors
# origin, never its own source of truth. See 04's note - `pull` refuses whenever an
# untracked results/ file would be overwritten, which is exactly what these
# notebooks create.
subprocess.run(['git','reset','-q','--hard', f'origin/{BRANCH}'], check=True)

# drop the token from the stored remote so it is not left on the VM's disk
subprocess.run(['git','remote','set-url','origin', bare_url], check=True)

!pip install -q -r requirements.txt
!git --no-pager log --oneline -3

**Prints** &nbsp; `all lambda sweep checks passed` plus the other suites. Anything else: stop here — the selection rule, the dev/test wall, and the one-forward-pass-per-pair cost model are all checked offline in seconds.

**Writes** &nbsp; Nothing.

In [ ]:
# offline checks first - seconds, no network, no GPU
!python -m tests.test_lambda_sweep | tail -3
!python -m tests.test_run_eval | tail -3
!python -m tests.test_data_pipeline | tail -3

## 2. The data this notebook is allowed to touch

Four files, and which stage may read which is the whole design:

| file | rows | stage |
|---|---:|---|
| `data/probe/splits/train.jsonl` | 152 | dev — lambda selection |
| `data/probe/hebrew_stsb_dev.csv` | 1,500 | dev — the STS trade-off guard |
| `data/probe/splits/test.jsonl` | 151 | test — final only |
| `data/probe/hebrew_stsb_test.csv` | 1,379 | test — final only |

All four are committed, so the clone above is enough — nothing to regenerate.

The cell below checks the two **dev** files only. The test files are left shut
until section 6: opening one to count its rows would not corrupt anything by
itself, but the habit is the thing being protected, and a row count is not worth
an exception to it.

**Prints** &nbsp; The two dev row counts, each confirmed against what it must be.

**Writes** &nbsp; Nothing. It only reads the files.

In [ ]:
from pathlib import Path
from src.harness import sts as sts_data
from src.harness.lambda_sweep import PROBE_TRAIN, PROBE_TEST
from src.schema import load_probe

def check_rows(expected):
    for path, n_expected in expected.items():
        n = (len(load_probe(path)) if path.endswith('.jsonl')
             else len(sts_data.load_sts(path)))
        assert n == n_expected, f'{path}: expected {n_expected}, found {n}'
        print(f'{path:38s} {n:5d} rows  ok')

check_rows({PROBE_TRAIN: 152, sts_data.DEV_PATH: 1500})

## 3. The NLI checkpoint on Drive

The model fine-tuned on **clean** HebNLI by `02_train_nli.ipynb` — the split that
excludes the 689 promptIDs the probe was mined from. The released
`oriel9p/AlephBERT-FT-HebNLI-LCHAIM` checkpoint saw those rows labelled
`contradiction` during its own training, so selecting lambda against it would be
tuning on data the classifier had memorised.

Read-only, and not retrained here.

**Prints** &nbsp; Drive's mount confirmation and a check that all four checkpoint files are there. No fallback: our checkpoint only exists on Drive.

**Writes** &nbsp; Nothing beyond mounting Drive at `/content/drive`.

In [ ]:
CKPT = '/content/drive/MyDrive/hebrew-negation/checkpoints/alephbert-hebnli-clean'

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    raise RuntimeError(
        'Drive did not mount. Our checkpoint only exists on Drive - make sure this '
        'session is signed in with the same Google account used for training. '
        f'({type(exc).__name__}: {exc})'
    ) from exc

ckpt_dir = Path(CKPT)
if not ckpt_dir.is_dir():
    raise RuntimeError(f'{CKPT} not found on Drive - check this mounted the right account')
for name in ('model.safetensors', 'config.json', 'tokenizer.json', 'tokenizer_config.json'):
    if not (ckpt_dir / name).exists():
        raise RuntimeError(f'{name} missing from {CKPT}')
print('checkpoint dir:', CKPT)

**Prints** &nbsp; The checkpoint path, `encoding pair`, the label names read from its config, six pairs each `[ok]`/`[MISMATCH]`, and a tally. Read this before trusting any number below: if entailment and contradiction were swapped, every `nli_score` would flip sign and nothing else would complain.

**Writes** &nbsp; Nothing.

In [ ]:
!python -m src.interventions.check_nli_labels --model {CKPT} --subfolder ""

## 4. Dev — sweep lambda and lock one value per model

`0.00, 0.05, ..., 1.00` on Probe-train and STS-dev, four embedders,
**independently**. Both halves of the blend are computed once per pair and the 21
grid points are arithmetic on the cached arrays; the NLI half does not depend on
the embedder either, so it is computed once and shared. That is 1,804 NLI forward
passes for the whole sweep instead of 151,536.

The rule, in one line: **among lambdas whose STS-dev Spearman stays within 0.02
of that model's own cosine-only Spearman, take the highest `pairwise_accuracy`;
break ties by `mean_gap`, then by the smallest lambda.** `lambda=0` is the
baseline and is always eligible, so a selection always exists.

Pearson is reported but takes no part in the rule — one correlation decides
eligibility, and it is the rank-based one, because STS gold scores are ordinal
judgements.

**Prints** &nbsp; The row counts, `[nli]` lines for the three pair sets (152 + 152 + 1,500), then one `[dev]` line per model with its selected lambda, accuracy against the cosine baseline, gap, and STS Spearman with its drop. Ten to twenty minutes on a T4, most of it downloading the four embedders.

**Writes** &nbsp; `results/nli_lambda_dev.csv` — 84 rows, the complete sweep.<br>`results/nli_selected_lambdas.json` — the four locked selections.

In [ ]:
!python -m src.harness.lambda_sweep --stage dev \
    --models multilingual-e5 labse alephbert-sentence sambert \
    --nli-model {CKPT} --nli-subfolder "" --nli-encoding pair

### What the sweep looks like

Two things to read off the table below. **Where accuracy stops improving** —
if it is flat from 0.4 upward, the tie-break to the smallest lambda is doing
real work and the choice is the conservative end of a plateau, not a peak.
**Where eligibility runs out** — if `eligible` goes False well before the best
accuracy, STS is the binding constraint and the selected lambda is a
compromise, which is worth saying plainly in the report.

**Prints** &nbsp; Per model: the selected row, then the eligible/ineligible counts and the accuracy at every grid point.

**Writes** &nbsp; Nothing. It only reads the csv back.

In [ ]:
import pandas as pd

dev = pd.read_csv('results/nli_lambda_dev.csv')
print(dev[dev.selected][['model','lambda','pairwise_accuracy','mean_gap',
                          'sts_spearman','sts_spearman_drop']].to_string(index=False))

print()
for model, group in dev.groupby('model'):
    eligible = group[group.eligible]
    print(f'{model:20s} eligible {len(eligible):2d}/21  '
          f'up to lambda={eligible["lambda"].max():.2f}')

dev.pivot_table(index='lambda', columns='model', values='pairwise_accuracy')

**Prints** &nbsp; The locked file, verbatim. Four models, one lambda each. **This is the point of no return** — everything below reads it and nothing below may change it.

**Writes** &nbsp; Nothing.

In [ ]:
import json

locked = json.load(open('results/nli_selected_lambdas.json', encoding='utf-8'))
print(json.dumps(locked, ensure_ascii=False, indent=2))
assert len(locked['selected']) == 4, 'expected exactly one lambda per model'

## 5. Commit the selection before running the test stage

Not housekeeping — this is what makes the ordering checkable by someone who was
not in the room. Download the two dev files now, commit them from your machine
with the `nli:` prefix, and only then run section 6. A selection committed after
the test numbers exist proves nothing about which came first.

If the session dies here, that is fine: section 6 re-reads the committed json.

**Prints** &nbsp; Browser downloads, or the files printed inline when that is unavailable — printed unconditionally, since under the VS Code extension `files.download()` can return without raising while the file lands nowhere findable.

**Writes** &nbsp; Your machine, as downloads and/or as text in the saved notebook.

In [ ]:
DEV_RESULTS = ['results/nli_lambda_dev.csv', 'results/nli_selected_lambdas.json']
try:
    from google.colab import files
    for path in DEV_RESULTS:
        files.download(path)
except Exception as exc:
    print(f'[warn] browser download unavailable ({type(exc).__name__})')

for path in DEV_RESULTS:
    print(f'\n===== {path} =====')
    print(open(path, encoding='utf-8').read())

## 6. Test — the locked evaluation, once

Probe-test (151 items) and STS-test (1,379 pairs), for two configurations per
model: `lambda=0` and that model's locked lambda. No grid, no selection. A model
whose selected lambda is 0 gets one row rather than the same run reported twice.

`run_test` reads `results/nli_selected_lambdas.json` and refuses to start if it
is missing, or if it was written under a different NLI checkpoint than the one
passed here — a lambda is only meaningful next to the classifier it was chosen
with. It has no code path that writes the json back.

**Run this once.** If a test number disappoints, that is a result, not a reason
to go back to section 4.

**Prints** &nbsp; The two test-split row counts — the first time this notebook opens either file.

**Writes** &nbsp; Nothing.

In [ ]:
check_rows({PROBE_TEST: 151, sts_data.TEST_PATH: 1379})

**Prints** &nbsp; The row counts, the `[nli]` lines again (151 + 151 + 1,379 — different pairs, so the classifier does run again), then one `[test]` line per row.

**Writes** &nbsp; `results/nli_lambda_test.csv` — the publication table, up to 8 rows.

In [ ]:
!python -m src.harness.lambda_sweep --stage test \
    --models multilingual-e5 labse alephbert-sentence sambert \
    --nli-model {CKPT} --nli-subfolder "" --nli-encoding pair

**Prints** &nbsp; The final table: per model, baseline against selected, on the probe and on STS.

**Writes** &nbsp; Nothing. It only reads the csv back.

In [ ]:
test = pd.read_csv('results/nli_lambda_test.csv')
print(test[['model','configuration','lambda','pairwise_accuracy','mean_gap',
            'sts_pearson','sts_spearman']].to_string(index=False))

# Did the selected lambda beat plain cosine on the held-out probe, and what did
# it cost on held-out STS? Both halves of the trade-off, per model.
wide = test.pivot_table(index='model', columns='configuration',
                        values=['pairwise_accuracy','sts_spearman'])
wide

## 7. Download the final results

Commit with the `nli:` prefix. `results/nli_lambda_dev.csv` and
`results/nli_selected_lambdas.json` should already be committed from section 5 —
if the diff shows them changing now, something re-ran the dev stage after the
test stage, and the test numbers below are no longer a held-out measurement.

**Prints** &nbsp; The final csv, inline and as a download.

**Writes** &nbsp; Your machine.

In [ ]:
FINAL = ['results/nli_lambda_test.csv']
try:
    from google.colab import files
    for path in FINAL:
        files.download(path)
except Exception as exc:
    print(f'[warn] browser download unavailable ({type(exc).__name__})')

for path in FINAL:
    print(f'\n===== {path} =====')
    print(open(path, encoding='utf-8').read())

!git status --short results/